# Model C_7: Topic modeling (BERTopic) sebelum ekstraksi fitur

**Tujuan (1b.ii).** Mengelompokkan artikel ke tema-tema geopolitik secara otomatis (tanpa target kurs), lalu
memakai tema itu untuk membentuk fitur. Tiga varian, masing-masing menjawab pertanyaan berbeda:

| Varian | Cara memakai topik | Pertanyaan |
|---|---|---|
| `C7_share` | porsi artikel per topik tiap hari, di-decay dalam jendela 20 hari | Apakah tahu *tema apa* yang mendominasi berita membantu? |
| `C7_filter` | hanya artikel di topik relevan yang dipakai untuk vektor FinBERT harian | Kategorisasi *sebelum* ekstraksi fitur (sesuai bunyi tugas) |
| `C7_chan` | topik menggantikan channel buatan tangan di varian C6 terbaik | Apakah tema hasil data sebaik channel yang kita rancang sendiri? |

**Desain supaya jumlah varian kecil**
- Hanya BERTopic (bukan LDA): embedding MiniLM sudah ada dari C6, dan LDA pada judul+lead yang pendek
  cenderung menghasilkan topik yang kabur.
- Clustering KMeans dengan `N_TOPICS` tetap (bukan HDBSCAN, yang sering menyisakan banyak artikel "outlier"
  dan jumlah topik yang berubah-ubah). `N_TOPICS` dipilih dengan melihat kata kunci topik di data train,
  BUKAN skor validation.
- Semua varian memakai HGB dengan setelan dari C6 (half-life, k novelty, τ) → `n_configs_tried = 1`.
- Pembanding tidak perlu dijalankan ulang: `C6_base_HGB` (vektor FinBERT sama rata, tanpa filter) dan
  hasil 1a terbaik (`C2_finbert_art_cls_HGB`) sudah ada di `results_c/`.

**Aturan.** BERTopic di-fit pada artikel **train saja**; artikel val/test hanya di-*assign* ke topik.
Daftar topik relevan ditentukan dari tabel topik (data train, tanpa target) lalu **dibekukan** sebelum
melihat skor validation.

**Prasyarat.** Jalankan `05f_C6_salience.ipynb` dulu: notebook ini memakai embedding kalimat yang disimpan
C6 (`c6_scorer_minilm.npy`, `c6_finbert_sent_cls.npy`) dan setelan terbaik C6 dari `results_c/C6_*_HGB.json`.
Butuh paket `bertopic` (otomatis memasang `umap-learn`). Jalankan dari folder `src/`.

In [9]:
from c_common import *
import c_common
import re, time
from sklearn.decomposition import PCA

D, text = load_all()                      # corpus B_2, sama dengan C_2 / C_6
TITLE, LEAD = "judul_gabungan", "lead_paragraph_gabungan"
print({s: len(d) for s, d in D.items()}, text.shape)

{'train': 825, 'val': 176, 'test': 178} (1202, 5)


## 1. Pengaturan

In [10]:
N_TOPICS = 15         
WINDOW = 20              
N_COMP = 20            
R_SHIFT = 1.0          
RELEVANT_TOPICS = None  
#Aturan: topik relevan = topik yang porsi artikel "menggerakkan dolar" (label dollar Qwen/C5 = stronger atau weaker, bukan neutral) >= median antar topik. Jadi kira-kira separuh topik terpilih.
#(Label event "other" tidak dipakai: korpus sudah difilter relevansi, jadi hampir tidak ada artikel "other".)

# Setelan dari C6: varian C6 terbaik (validation F1-macro) di antara varian yang memakai channel.
C6_V = ["base", "rel", "chan", "nov", "hier"]
USES = {"base": (False, False, False, False), "rel": (True, False, False, False),
        "chan": (True, True, False, False), "nov": (True, True, True, False), "hier": (True, True, True, True)}
_c6 = {}
for v in C6_V:
    f = RESULTS_C / f"C6_{v}_HGB.json"
    assert f.exists(), f"{f} tidak ada: jalankan 05f_C6_salience.ipynb dulu"
    _c6[v] = json.loads(f.read_text())
REF_V = max(["chan", "nov", "hier"], key=lambda v: _c6[v]["val"]["F1_macro"])
REF = _c6[REF_V]["c6_config"]                    
print("varian C6 acuan:", REF_V, REF, "| F1 val:", round(_c6[REF_V]["val"]["F1_macro"], 3))

varian C6 acuan: nov {'tau': 1.0, 'k': 5, 'half': 2} | F1 val: 0.545


## 2. Kalimat, artikel, dan embedding dari C6

Tabel kalimat dibangun dengan fungsi yang **sama persis** dengan C6 supaya urutannya cocok dengan cache
embedding. Embedding artikel = rata-rata embedding kalimatnya (MiniLM untuk topik, FinBERT untuk representasi).

In [11]:
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'“])")


def build_sentence_table(text_df):
    rows, art_id, fallback = [], 0, 0
    for pos, (t, l) in enumerate(zip(text_df[TITLE].fillna(""), text_df[LEAD].fillna(""))):
        titles = [x.strip() for x in t.split(" | ") if x.strip()]
        if not titles:
            continue
        leads = [x.strip() for x in l.split(" | ")]
        if len(leads) != len(titles):
            leads, fallback = [""] * len(titles), fallback + 1
        for ti, li in zip(titles, leads):
            sents, seen = [], set()
            for s in [ti] + [x.strip() for x in SENT_SPLIT.split(li)]:
                key = s.lower().strip(" .")
                if len(s.split()) >= 3 and key not in seen:
                    seen.add(key)
                    sents.append(s)
            if not sents:
                sents = [ti]
            rows += [(pos, art_id, j, s) for j, s in enumerate(sents)]
            art_id += 1
    S = pd.DataFrame(rows, columns=["day", "art", "sent_no", "sentence"])
    S["date"] = text_df.index[S["day"].values]
    split_of = {d: s for s in D for d in D[s]["date"]}
    S["split"] = S["date"].map(split_of).fillna("warmup")
    return S


S = build_sentence_table(text)
E_sc = np.load(FEATURES_C / "c6_scorer_minilm.npy")
E_rep = np.load(FEATURES_C / "c6_finbert_sent_cls.npy")
assert len(E_sc) == len(S) == len(E_rep), "cache C6 tidak cocok dengan tabel kalimat: jalankan ulang 05f"

DAY, ART = S["day"].values, S["art"].values
N_DAYS = len(text)
is_tr = (S["split"] == "train").values


def group_mean(keys, vecs):
    return pd.DataFrame(vecs).groupby(keys).mean()


A = S.groupby("art").agg(day=("day", "first"), date=("date", "first"), split=("split", "first"),
                         doc=("sentence", " ".join))
A_sc = group_mean(ART, E_sc).loc[A.index].values
A_sc = A_sc / (np.linalg.norm(A_sc, axis=1, keepdims=True) + 1e-9)      #embedding artikel untuk BERTopic
A_rep = group_mean(ART, E_rep).loc[A.index].values               #vektor FinBERT artikel 
a_tr = (A["split"] == "train").values
print(len(A), "artikel |", int(a_tr.sum()), "artikel train")

11764 artikel | 7081 artikel train


## 3. BERTopic (fit pada artikel train saja)

In [12]:
from bertopic import BERTopic
from umap import UMAP
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

t0 = time.time()
topic_model = BERTopic(
    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED),
    hdbscan_model=KMeans(n_clusters=N_TOPICS, n_init=10, random_state=SEED),   
    vectorizer_model=CountVectorizer(stop_words="english", min_df=5, ngram_range=(1, 2)),
    verbose=False,
)
docs = A["doc"].tolist()
topics_tr, _ = topic_model.fit_transform([docs[i] for i in np.where(a_tr)[0]], embeddings=A_sc[a_tr])
topic_of = np.empty(len(A), dtype=int)
topic_of[a_tr] = topics_tr                              
topic_of[~a_tr], _ = topic_model.transform([docs[i] for i in np.where(~a_tr)[0]], embeddings=A_sc[~a_tr])
A["topic"] = topic_of
print(f"BERTopic selesai ({time.time() - t0:.0f} dtk)")

W1119 10:49:36.339000 15704 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


BERTopic selesai (266 dtk)


## 4. Tabel topik & pemilihan topik relevan (train saja, tanpa target)

Untuk tiap topik: jumlah artikel train, kata kunci, contoh judul, dan label event mayoritas dari Qwen (C5).
Aturan otomatis: topik relevan kalau porsi artikel yang menurut Qwen menggerakkan dolar (`porsi_dollar`,
label dollar ≠ neutral) ≥ median antar topik. Kalau tidak setuju, isi
`RELEVANT_TOPICS` secara manual di bagian Pengaturan, lalu jalankan ulang dari sini. **Bekukan sebelum
melihat skor validation.**

In [13]:
pd.set_option("display.max_colwidth", 200)
info = []
for t in range(N_TOPICS):
    words = [w for w, _ in topic_model.get_topic(t)][:6]
    tr_t = A[a_tr & (topic_of == t)]
    info.append({"topic": t, "n_train": len(tr_t), "kata_kunci": ", ".join(words),
                 "contoh_judul": " || ".join(s.split(". ")[0][:80] for s in tr_t["doc"].head(3))})
TOPICS = pd.DataFrame(info).set_index("topic")

f_c5 = FEATURES_C / "c5_llm_labels.jsonl"
if f_c5.exists():
    c5 = pd.read_json(f_c5, lines=True)
    c5 = c5[c5["valid"]].set_index("id")
    lab = A[a_tr].join(c5[["event", "dollar"]], how="inner")
    TOPICS["c5_event_mayoritas"] = lab.groupby("topic")["event"].agg(lambda x: x.value_counts().index[0])
    TOPICS["porsi_dollar"] = lab.groupby("topic")["dollar"].apply(lambda x: (x != "neutral").mean()).round(2)

if RELEVANT_TOPICS is None:
    assert "porsi_dollar" in TOPICS, "label C5 tidak ada: isi RELEVANT_TOPICS secara manual"
    REL = sorted(TOPICS.index[TOPICS["porsi_dollar"] >= TOPICS["porsi_dollar"].median()].tolist())
    print("topik relevan (aturan otomatis C5):", REL)
else:
    REL = sorted(RELEVANT_TOPICS)
    print("topik relevan (manual):", REL)
IRR = [t for t in range(N_TOPICS) if t not in REL]
assert REL and IRR, "butuh minimal satu topik relevan dan satu tidak relevan"
TOPICS["relevan"] = TOPICS.index.isin(REL)
TNAME = {t: f"t{t}_" + "_".join(re.sub(r"\W+", "", w) for w, _ in topic_model.get_topic(t)[:2]) for t in range(N_TOPICS)}
display(TOPICS.sort_values("porsi_dollar", ascending=False) if "porsi_dollar" in TOPICS else TOPICS)
a_rel = np.isin(topic_of, REL)
print(f"artikel train di topik relevan: {a_rel[a_tr].mean():.0%}")

topik relevan (aturan otomatis C5): [0, 1, 3, 4, 8, 11, 12, 13, 14]


,n_train,kata_kunci,contoh_judul,c5_event_mayoritas,porsi_dollar,relevan
topic,,,,,,
11,273,"dollar, yen, bank, euro, currency, data",U.S || Dollar index dips to session low following Federal Reserve statement The dollar || Dollar dips but shows biggest weekly gain in almost 3 months The U.S,monetary policy,0.90,True
13,124,"gold, spot, futures, prices, dollar, federal","Gold jumps 2% as dollar slips, but set for quarterly fall Gold prices rose more || Gold prices gain as inflation concerns grow Gold prices rose on Tuesday, as risi || Gold is set to rally the com...",economic data,0.76,True
14,89,"treasury, yield, 10year, 10year treasury, treasury yield, basis",Treasury yields gain after retail sales increase in September Treasury yields ro || 10-year Treasury yield starts the week higher with focus on inflation data The 1 || U.S,economic data,0.67,True
3,655,"rate, rates, federal, fed, inflation, reserve","Fed Chair Powell calls inflation 'frustrating' and sees it running into next yea || Even with low interest rates, mortgage payments are increasingly unaffordable Ev || ECB hawk and German central ...",monetary policy,0.61,True
12,235,"tariffs, electric, tariff, trump, vehicles, car",EU working to avoid new tariffs on the U.S || Bourbon distillers face big tax bills and higher tariffs after a record year for || GM Defense plans to produce military vehicle based on Hummer EV in...,trade/tariffs,0.53,True
4,589,"oil, crude, energy, prices, barrel, oil prices","Exxon Mobil has been lobbying against parts of Democrats' big social and climate || Cathie Wood compares current crude market to whale oil, predicts it will meet th || Amplify Energy shares crash ...",oil/energy,0.51,True
1,906,"ukraine, russian, russia, russias, war, putin","‘Russianization’ of China? French military think tank says Beijing borrowing fro || Havana Syndrome: ‘Attacks have stepped-up in their brazenness,’ says national se || 5 charts show Russia's econo...",war/conflict,0.36,True
8,435,"israel, gaza, israeli, hamas, sea, red","Mayor of Kabul warns, ‘We have millions of people who are at the verge of famine || The Taliban takeover of Afghanistan could reshape counterinsurgencies in Africa, || Investigators board ship tha...",war/conflict,0.34,True
0,909,"inflation, prices, labor, consumer, price, year",'This is not 1973': Economist rules out 'stagflation' and persistent price press || Chamath Palihapitiya is very worried about inflation and hedging in 3 ways Inves || New York Life Investments CE...,economic data,0.34,True


artikel train di topik relevan: 60%


## 5. Fitur harian per varian

- **share**: porsi artikel hari itu di tiap topik (`N_TOPICS` fitur) + volume.
- **filter**: vektor FinBERT harian = rata-rata artikel di topik relevan saja (sama rata, seperti `C6_base`,
  jadi perbedaannya dengan `C6_base` murni efek filter), PCA 20 + jumlah artikel relevan/semua.
- **chan**: mesin C6 dengan channel = topik relevan. Skor kalimat ke topik = kemiripan ke centroid topik
  (rata-rata embedding artikel train di topik itu), distandardisasi dengan statistik train lalu sigmoid.
  Topik tidak relevan berperan sebagai channel *noise*. Bobot, novelty, dan pooling mengikuti varian C6 acuan.

In [14]:
def to_days(df_by_day, n_cols):
    out = np.zeros((N_DAYS, n_cols), dtype="float64")
    out[df_by_day.index.values] = df_by_day.values
    return out


def group_wmean(keys, vecs, w):
    num = pd.DataFrame(vecs * w[:, None]).groupby(keys).sum()
    return num.div(pd.Series(w).groupby(keys).sum(), axis=0)


def pca_days(V, has, prefix="v_"):
    #PCA di-fit pada hari train yang punya data, hari tanpa data = 0.
    tr_pos = text.index.get_indexer(D["train"]["date"])
    tr_pos = tr_pos[has[tr_pos]]
    Vp = PCA(n_components=N_COMP, random_state=SEED).fit(V[tr_pos]).transform(V)
    Vp[~has] = 0
    return pd.DataFrame(Vp, index=text.index, columns=[f"{prefix}{i}" for i in range(N_COMP)])


has_news = np.zeros(N_DAYS, bool)
has_news[A["day"].unique()] = True
n_art = to_days(A.groupby("day").size().to_frame(), 1)[:, 0]


def feats_share():
    oh = pd.get_dummies(A["topic"]).reindex(columns=range(N_TOPICS), fill_value=0).astype(float)
    F = pd.DataFrame(to_days(oh.groupby(A["day"].values).mean(), N_TOPICS), index=text.index,
                     columns=[f"tp_{TNAME[t]}" for t in range(N_TOPICS)])
    F["has_news"] = has_news.astype(float)
    F["log_n_art"] = np.log1p(n_art)
    return F, "has_news"


def feats_filter():
    keep = a_rel
    day_vec = pd.DataFrame(A_rep[keep]).groupby(A["day"].values[keep]).mean()
    has_rel = np.zeros(N_DAYS, bool)
    has_rel[day_vec.index.values] = True
    F = pca_days(to_days(day_vec, A_rep.shape[1]), has_rel)
    F["has_rel"] = has_rel.astype(float)
    F["has_news"] = has_news.astype(float)
    F["log_n_rel"] = np.log1p(to_days(A[keep].groupby("day").size().to_frame(), 1)[:, 0])
    F["log_n_art"] = np.log1p(n_art)
    return F, "has_rel"


#skor kalimat ke topik (untuk varian chan)
CENT = np.stack([A_sc[a_tr & (topic_of == t)].mean(0) for t in range(N_TOPICS)])
CENT = CENT / np.linalg.norm(CENT, axis=1, keepdims=True)
RAW = E_sc @ CENT.T
Z = (RAW - RAW[is_tr].mean(0)) / (RAW[is_tr].std(0) + 1e-8)
P = 1 / (1 + np.exp(-(Z - R_SHIFT)))
Z_rel, P_rel = Z[:, REL], P[:, REL]
P_noise = P[:, IRR].max(1)
relevance = P_rel.max(1) * (1 - P_noise)


def novelty(E, day, k):
    nov = np.ones(len(E), dtype="float32")
    by_day = {d: np.where(day == d)[0] for d in np.unique(day)}
    for d, ix in by_day.items():
        prev = [by_day[p] for p in range(d - k, d) if p in by_day]
        if prev:
            pv = np.concatenate(prev)
            nov[ix] = 1 - (E[ix] @ E[pv].T).max(1)
    return np.clip(nov, 0, 1)


def feats_chan():
    _, _, use_nov, hier = USES[REF_V]
    tau, k = REF["tau"], REF["k"]
    nov = novelty(E_sc, DAY, k) if use_nov else None
    w = relevance * (nov if use_nov else 1) + 1e-6
    if hier:
        art_vec = group_wmean(ART, E_rep, w)
        art_w = pd.Series(w).groupby(ART).max().loc[art_vec.index].values
        art_day = pd.Series(DAY).groupby(ART).first().loc[art_vec.index].values
        day_vec = group_wmean(art_day, art_vec.values, art_w)
    else:
        day_vec = group_wmean(DAY, E_rep, w)
    F = pca_days(to_days(day_vec, E_rep.shape[1]), has_news)
    e = np.exp((Z_rel - Z_rel.max(1, keepdims=True)) / tau)
    wc = w[:, None] * e / e.sum(1, keepdims=True)
    names = [TNAME[t] for t in REL]
    F[[f"ch_{n}_mass" for n in names]] = np.log1p(to_days(pd.DataFrame(wc).groupby(DAY).sum(), len(REL)))
    F[[f"ch_{n}_max" for n in names]] = to_days(pd.DataFrame(wc).groupby(DAY).max(), len(REL))
    if use_nov:
        nv = pd.Series(nov)
        F["nov_mean"] = to_days(nv.groupby(DAY).mean().to_frame(), 1)[:, 0]
        F["nov_max"] = to_days(nv.groupby(DAY).max().to_frame(), 1)[:, 0]
        art_nov = nv.groupby(ART).mean()
        thr = art_nov[a_tr].median()
        F["nov_share_new"] = to_days((art_nov > thr).groupby(A["day"].values).mean().to_frame(), 1)[:, 0]
    F["has_news"] = has_news.astype(float)
    F["log_n_art"] = np.log1p(n_art)
    return F, "has_news"


FEATS = {"share": feats_share, "filter": feats_filter, "chan": feats_chan}

In [15]:
#Cek kewajaran varian chan 
for j, t in enumerate(REL):
    sc = P_rel[is_tr, j] * (1 - P_noise[is_tr])
    top = S[is_tr].assign(score=sc).nlargest(3, "score")
    print(f"\n=== {TNAME[t]}")
    for _, r in top.iterrows():
        print(f"   {r.score:.2f}  {r.sentence[:110]}")


=== t0_inflation_prices
   0.46  Consumers saw inflation pick up slightly in November as price increases in categories including groceries, gas
   0.46  The Labor Department's consumer inflation report showed that prices are still jumping.
   0.45  Inflation ticked up in December on the back of higher energy and food prices, the Bureau of Labor Statistics r

=== t1_ukraine_russian
   0.75  Russian forces launched an attack on Ukraine early Thursday morning local time, marking the devastating culmin
   0.74  Russian forces invade Ukraine
   0.74  How Wagner's short-lived revolt could shape the direction of Russia's war in Ukraine

=== t3_rate_rates
   0.53  The Federal Reserve announced Wednesday it will leave interest rates unchanged as inflation continues to prove
   0.52  The Federal Reserve is expected to raise rates Wednesday as they look to contain soaring inflation.
   0.52  Federal Reserve Governor Christopher Waller said Thursday that the central bank could lower interest rate

## 6. Jendela waktu & evaluasi (HGB, setelan C6)

Jendela t−19..t dengan half-life dari C6 acuan, cara rata-rata sama dengan C6: fitur kondisional
(vektor, porsi topik, novelty) dirata-rata atas hari yang punya data, fitur intensitas atas semua hari.

In [16]:
COND_PREFIX = ("v_", "nov_", "tp_")


def window_decay(F, half, has_col, window=WINDOW):
    cond = [c for c in F.columns if c.startswith(COND_PREFIX)]
    inten = [c for c in F.columns if c not in cond]
    Xc, Xi, has = F[cond].values, F[inten].values, F[has_col].values
    n = len(F)
    num_c, den_c = np.zeros_like(Xc), np.zeros(n)
    num_i, den_i = np.zeros_like(Xi), np.zeros(n)
    for L in range(window):
        wl = 0.5 ** (L / half)
        num_c[L:] += wl * has[:n - L, None] * Xc[:n - L]
        den_c[L:] += wl * has[:n - L]
        num_i[L:] += wl * Xi[:n - L]
        den_i[L:] += wl
    out = pd.DataFrame(np.where(den_c[:, None] > 0, num_c / np.maximum(den_c, 1e-12)[:, None], 0.0),
                       index=F.index, columns=["dk_" + c for c in cond])
    out[["dk_" + c for c in inten]] = num_i / den_i[:, None]
    out["dk_news_frac"] = den_c / den_i
    return out


def build(v):
    F, has_col = FEATS[v]()
    W = window_decay(F, REF["half"], has_col)
    Dx = {s: D[s].merge(W, left_on="date", right_index=True, how="left").fillna(0) for s in D}
    return Dx, list(W.columns)


for v in FEATS:
    Dx, cols = build(v)
    name = f"C7_{v}_HGB"
    note = (f"C7 BERTopic (KMeans {N_TOPICS} topik, fit train), varian {v}; relevan={REL}; "
            f"setelan C6 {REF_V} {REF}")
    evaluate_val(name, "HGB", Dx, PRICE + cols, n_configs_tried=1, note=note)
    p = json.loads((RESULTS_C / f"{name}.json").read_text())["best_params"]
    prob = make_model("HGB", p).fit(Dx["train"][PRICE + cols], Dx["train"]["target"]) \
                               .predict_proba(Dx["val"][PRICE + cols])[:, 1]
    c_common._save(name, {"c7_config": {"n_topics": N_TOPICS, "relevant": REL, "c6_ref": REF_V, **REF},
                          "val_auc_ci95": bootstrap_auc_ci(Dx["val"]["target"].values, prob)})

C7_share_HGB [HGB] val: DA=0.540, BalAcc=0.535, F1_macro=0.527, MCC=0.073, AUC=0.496
C7_filter_HGB [HGB] val: DA=0.489, BalAcc=0.490, F1_macro=0.489, MCC=-0.021, AUC=0.454
C7_chan_HGB [HGB] val: DA=0.517, BalAcc=0.515, F1_macro=0.514, MCC=0.030, AUC=0.439


## 7. Ringkasan (validation) dengan pembanding

In [17]:
res = collect_results()
ref_names = ["C2_finbert_art_cls_HGB", "C6_base_HGB", f"C6_{REF_V}_HGB"]
res = res[res["name"].str.startswith("C7_") | res["name"].isin(ref_names)].copy()
res["val_auc_ci95"] = [json.loads((RESULTS_C / f"{n}.json").read_text()).get("val_auc_ci95") for n in res["name"]]
show = ["name", "n_features", "n_configs_tried", "val_DA", "val_F1_macro", "val_MCC", "val_AUC", "val_auc_ci95"]
display(res[show].round(3))
print("Perbandingan yang penting: C7_filter vs C6_base (efek filter topik), "
      f"C7_chan vs C6_{REF_V} (topik vs channel buatan tangan), C7_share vs C2 (tema vs embedding).")

,name,n_features,n_configs_tried,val_DA,val_F1_macro,val_MCC,val_AUC,val_auc_ci95
2,C2_finbert_art_cls_HGB,32,16,0.591,0.588,0.179,0.568,None
31,C6_base_HGB,34,3,0.551,0.551,0.102,0.513,"[0.414, 0.595]"
40,C6_nov_HGB,49,12,0.545,0.545,0.095,0.515,"[0.436, 0.589]"
45,C7_chan_HGB,55,1,0.517,0.514,0.030,0.439,"[0.348, 0.543]"
46,C7_filter_HGB,36,1,0.489,0.489,-0.021,0.454,"[0.36, 0.546]"
47,C7_share_HGB,29,1,0.540,0.527,0.073,0.496,"[0.421, 0.587]"


Perbandingan yang penting: C7_filter vs C6_base (efek filter topik), C7_chan vs C6_nov (topik vs channel buatan tangan), C7_share vs C2 (tema vs embedding).


## 8. Kontrol teks diacak & tes akhir (SEKALI)

Pilih satu varian C7 berdasarkan validation (atau tidak sama sekali kalau tidak ada yang di atas pembanding),
jalankan kontrol, lalu tes sekali.

In [18]:
CHOSEN = None     
RUN_TEST = False     

if CHOSEN is not None:
    Dx, cols = build(CHOSEN)
    rec = json.loads((RESULTS_C / f"C7_{CHOSEN}_HGB.json").read_text())
    print("kontrol teks diacak:", shuffled_text_control("HGB", Dx, PRICE, cols, rec["best_params"]))
    print("AUC val asli:", round(rec["val"]["AUC"], 3))
    if RUN_TEST:
        final_test(f"C7_{CHOSEN}_HGB", "HGB", Dx, PRICE + cols)